In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

In [ ]:
df = pd.read_csv("istanbulApartmentForRent.csv")

In [ ]:
df = df[(df["area (m2)"] < 400) & (df["price"] < 500000)]
df = df[(df["price"] > 5000) & (df["area (m2)"] > 0)]
df = df.dropna()

In [ ]:
df["area_per_room"] = df["area (m2)"] / df["room"]
df["age_area_ratio"] = df["area (m2)"] / (df["age"] + 1)
df["age_area_interaction"] = df["age"] * df["area (m2)"]
df["room_density"] = df["room"] / df["area (m2)"]

'''
df["price_per_m2"] = df["price"] / df["area (m2)"]

alt_sinir = df["price_per_m2"].quantile(0.05)
ust_sinir = df["price_per_m2"].quantile(0.95) 

df = df[df["price_per_m2"]<ust_sinir]
df = df[df["price_per_m2"]>alt_sinir]''' 
#without price_per_m2 the score is 0.68 and price_per_m2 makes it 0.99 reliable which is not realistic 

#df['age_category'] = pd.cut(df['age'], bins=[-1, 5, 15, 100], labels=[0, 1, 2]) this restriction decreases the score

In [ ]:
x = df[["area (m2)", "age", "room", "area_per_room", "age_area_ratio","age_area_interaction","room_density"]]
y = df["price"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
model = XGBRegressor(
    n_estimators=500,    
    learning_rate=0.01,    
    max_depth=4,           
    subsample=0.8,
    enable_categorical=True,
    random_state=42
)
model.fit(x_train, y_train)

In [ ]:
sns.scatterplot(data=df,x="area (m2)",y="price")

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(x_test['area (m2)'], y_test, color='royalblue', alpha=0.5, label='Actual Data')

plt.scatter(x_test['area (m2)'], model.predict(x_test), color='red', alpha=0.5, label='Model Prediction')

plt.title(f'Multiple Regression Model (R2 Score: {skor:.2f})', fontsize=15)
plt.xlabel('Area (m2)', fontsize=12)
plt.ylabel('Price', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.show()

In [ ]:
skor = model.score(x_test, y_test)
print(f"v2.0 XGBoost Success Rate: {skor:.4f}")